# 03 — Heat Head Adaptation

Adapts the pretrained flow encoder to predict heat transfer for ALD showerhead impingement:
- **Nu** — Nusselt number (dimensionless heat transfer coefficient)
- **h** — convective heat transfer coefficient [W/m²K]
- **T_norm** — normalised wall temperature proxy

**Strategy:**
1. Load pretrained flow encoder from `02_pretrain_flow`
2. Freeze encoder — only train the heat decoder head
3. Generate synthetic jet impingement dataset (Martin 1977 correlation)
4. Train heat head on synthetic data
5. Fine-tune encoder + heat head together at low LR

**Requires:** A100 GPU

## 0. Install + setup

In [ ]:
import subprocess, torch
subprocess.run(['pip', 'install', '-q', 'torch_geometric', 'h5py', 'scipy', 'tqdm'], check=True)
print('Done.')

In [ ]:
import os, sys, json, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = '/content/drive/MyDrive/cfd-ald-app'
except ImportError:
    BASE_DIR = os.path.expanduser('~/cfd-ald-app-data')

REPO_URL = 'https://github.com/Ranaam21/cfd-ald-app.git'
REPO_DIR = '/content/cfd-ald-app'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(r.stdout.strip() or 'Repo up to date.')
else:
    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'git clone failed: {r.stderr}')
    print('Cloned.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

FLOW_CKPT_DIR = Path(BASE_DIR) / 'checkpoints' / 'flow_head'
HEAT_CKPT_DIR = Path(BASE_DIR) / 'checkpoints' / 'heat_head'
HEAT_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Flow checkpoints → {FLOW_CKPT_DIR}')
print(f'Heat checkpoints → {HEAT_CKPT_DIR}')

## 1. Imports + GPU check

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Tuple
from tqdm import tqdm
from scipy.spatial import cKDTree
from torch_geometric.nn import MessagePassing

assert torch.cuda.is_available(), 'GPU required'
DEVICE = torch.device('cuda')
print(f'PyTorch {torch.__version__}  |  {torch.cuda.get_device_name(0)}')

## 2. Config

In [ ]:
CFG = {
    # Model (must match flow head)
    'input_dim_nodes': 17,
    'input_dim_edges': 4,
    'hidden_dim':      256,
    'processor_size':  15,
    'flow_output_dim': 4,    # u_x, u_y, p, nu_t
    'heat_output_dim': 3,    # Nu, h, T_norm

    # Synthetic dataset
    'n_cases':         2000, # synthetic jet impingement cases
    'n_points':        512,  # spatial points per case
    'Re_range':        (100, 8000),
    'Pr':              0.71,  # air
    'H_D_range':       (2.0, 12.0),   # nozzle height / diameter
    'S_D_range':       (3.0, 8.0),    # nozzle pitch / diameter

    # Phase 1: heat head only (encoder frozen)
    'epochs_frozen':   60,
    'lr_frozen':       1e-3,

    # Phase 2: fine-tune everything
    'epochs_finetune': 40,
    'lr_finetune':     1e-4,
    'lr_encoder':      1e-5,  # much lower for encoder

    'grad_clip':       1.0,
    'bf16':            True,
    'save_every':      20,
    'log_every':       10,
}
print('Config ready.')

## 3. Model definition (same as flow head)

In [ ]:
def mlp(in_dim: int, out_dim: int, hidden: int = 256, layers: int = 2) -> nn.Sequential:
    dims = [in_dim] + [hidden] * (layers - 1) + [out_dim]
    mods = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods.append(nn.SiLU())
    return nn.Sequential(*mods)


class MGNProcessor(MessagePassing):
    def __init__(self, hidden: int):
        super().__init__(aggr='sum')
        self.edge_mlp = mlp(3 * hidden, hidden)
        self.node_mlp = mlp(2 * hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)

    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        N = x.size(0)
        new_e = self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], dim=-1))
        new_e = self.edge_norm(new_e + edge_attr)
        agg   = self.propagate(edge_index, x=x, edge_attr=new_e, size=(N, N))
        new_x = self.node_mlp(torch.cat([x, agg], dim=-1))
        new_x = self.node_norm(new_x + x)
        return new_x, new_e

    def message(self, edge_attr):
        return edge_attr


class MeshGraphNet(nn.Module):
    """Shared encoder + swappable decoder head."""

    def __init__(self, node_dim, edge_dim, out_dim, hidden=256, n_layers=15):
        super().__init__()
        self.node_enc = mlp(node_dim, hidden)
        self.edge_enc = mlp(edge_dim, hidden)
        self.processors = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.decoder = mlp(hidden, out_dim, hidden)

    def encode(self, x, edge_index, edge_attr):
        """Return latent node embeddings (before decoder)."""
        x = self.node_enc(x)
        e = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, e = proc(x, edge_index, e)
        return x

    def forward(self, x, edge_index, edge_attr):
        return self.decoder(self.encode(x, edge_index, edge_attr))


print('Model classes defined.')

## 4. Load pretrained flow encoder

In [ ]:
# Load flow head final checkpoint
flow_ckpt_path = FLOW_CKPT_DIR / 'flow_head_final.pt'
assert flow_ckpt_path.exists(), f'Flow checkpoint not found: {flow_ckpt_path}\nRun 02_pretrain_flow.ipynb first.'

flow_ckpt = torch.load(flow_ckpt_path, map_location='cpu')
print(f'Loaded flow checkpoint (epoch {flow_ckpt["epoch"]+1})')
print(f'  Train MSE: {flow_ckpt["final_train_loss"]:.4f}')
print(f'  Val MSE:   {flow_ckpt["final_val_loss"]:.4f}')

# Build encoder+flow_decoder, load weights
flow_model = MeshGraphNet(
    node_dim=CFG['input_dim_nodes'], edge_dim=CFG['input_dim_edges'],
    out_dim=CFG['flow_output_dim'], hidden=CFG['hidden_dim'], n_layers=CFG['processor_size'],
)
flow_model.load_state_dict(flow_ckpt['model'])

# Build heat model — same encoder, new heat decoder
heat_model = MeshGraphNet(
    node_dim=CFG['input_dim_nodes'], edge_dim=CFG['input_dim_edges'],
    out_dim=CFG['heat_output_dim'], hidden=CFG['hidden_dim'], n_layers=CFG['processor_size'],
)

# Copy encoder weights from flow model
heat_model.node_enc.load_state_dict(flow_model.node_enc.state_dict())
heat_model.edge_enc.load_state_dict(flow_model.edge_enc.state_dict())
heat_model.processors.load_state_dict(flow_model.processors.state_dict())
# heat_model.decoder is freshly initialised (heat-specific)

heat_model = heat_model.to(DEVICE)
del flow_model, flow_ckpt

n_params = sum(p.numel() for p in heat_model.parameters())
print(f'\nHeat model loaded: {n_params/1e6:.1f}M params (encoder pretrained, decoder fresh)')

## 5. Synthetic jet impingement dataset

Generates cases using the **Martin (1977)** confined jet impingement correlation:
```
Nu = G(r/D, H/D) × Re^0.67 × Pr^0.42
```
Each case: random (Re, H/D, S/D) → spatial Nu(r/D) distribution over a radial profile.

In [ ]:
def martin_nu(r_over_D: np.ndarray, Re: float, Pr: float,
              H_D: float, S_D: float) -> np.ndarray:
    """
    Martin (1977) jet impingement Nusselt number correlation.
    r_over_D: radial distance / nozzle diameter [array]
    Returns Nu at each radial position.
    """
    # Nozzle area fraction
    A_f = np.pi / (4 * S_D**2)

    # Radial profile function G(r/D, H/D)
    # Primary peak at stagnation, secondary dip at r/D ~ 1.5-2
    r = np.clip(r_over_D, 0.1, 10.0)
    G = (1 - 1.1 * r / (1 + 0.1 * (H_D - 6) * r)) * np.exp(-0.04 * r**1.4)
    G = np.clip(G, 0.01, 1.0)

    # Nozzle geometry factor
    F = 0.5 + 0.0072 * (6 - H_D)**2

    Nu = F * G * (Re**0.67) * (Pr**0.42) * (A_f**0.08)
    return Nu.astype(np.float32)


def generate_heat_case(n_points: int, Re_range, Pr: float,
                       H_D_range, S_D_range) -> dict:
    """Generate one synthetic jet impingement case."""
    Re  = float(np.random.uniform(*Re_range))
    H_D = float(np.random.uniform(*H_D_range))
    S_D = float(np.random.uniform(*S_D_range))

    # Radial points on impingement plate: r/D from 0 to 6
    r_D = np.random.uniform(0.0, 6.0, n_points).astype(np.float32)
    r_D.sort()

    # 2D coordinates: spread on impingement plate (z=0)
    theta  = np.random.uniform(0, 2*np.pi, n_points).astype(np.float32)
    coords = np.stack([
        r_D * np.cos(theta),
        r_D * np.sin(theta),
        np.zeros(n_points, dtype=np.float32),
    ], axis=1)   # [N, 3]

    # Node features: broadcast case params to each node
    # [Re_norm, Pr, H_D_norm, S_D_norm, r_D, theta, 0, 0, 0, 0, 0] → dim 11
    # Plus coords(3) + node_feats(3) = 17 total (same as flow head)
    Re_norm  = Re  / 5000.0
    HD_norm  = H_D / 8.0
    SD_norm  = S_D / 6.0
    node_feats = np.column_stack([
        np.full(n_points, Re_norm,  dtype=np.float32),
        np.full(n_points, Pr,       dtype=np.float32),
        r_D / 6.0,
    ])   # [N, 3]

    # Global features: [Re, Pr, H_D, S_D, 0, 0, 0, 0, 0, 0, 0] → dim 11
    global_f = np.array([Re/8000, Pr, H_D/12, S_D/8,
                          0, 0, 0, 0, 0, 0, 0], dtype=np.float32)
    global_broadcast = np.tile(global_f, (n_points, 1))   # [N, 11]

    node_input = np.concatenate([coords, node_feats, global_broadcast], axis=1)  # [N, 17]

    # Heat targets
    Nu    = martin_nu(r_D, Re, Pr, H_D, S_D)      # [N]
    k_air = 0.026  # W/mK
    D     = 0.002  # 2mm nozzle diameter
    h     = Nu * k_air / D                          # [N]
    T_norm = 1.0 / (Nu / Nu.max() + 1e-6)          # proxy: hotter where Nu low
    T_norm = (T_norm - T_norm.mean()) / (T_norm.std() + 1e-6)

    targets = np.stack([Nu, h, T_norm], axis=1).astype(np.float32)  # [N, 3]

    return {'coords': coords, 'node_input': node_input,
            'targets': targets, 'Re': Re, 'H_D': H_D, 'S_D': S_D}


# Generate full dataset
print(f'Generating {CFG["n_cases"]} synthetic jet impingement cases...')
np.random.seed(42)
all_cases = [
    generate_heat_case(CFG['n_points'], CFG['Re_range'], CFG['Pr'],
                       CFG['H_D_range'], CFG['S_D_range'])
    for _ in tqdm(range(CFG['n_cases']))
]

split = int(0.9 * CFG['n_cases'])
train_cases = all_cases[:split]
val_cases   = all_cases[split:]
print(f'Train: {len(train_cases)}  Val: {len(val_cases)}')

# Quick sanity check
c = all_cases[0]
Nu_ex = c['targets'][:, 0]
print(f'Example case: Re={c["Re"]:.0f}, H/D={c["H_D"]:.1f}, S/D={c["S_D"]:.1f}')
print(f'  Nu range: {Nu_ex.min():.1f} – {Nu_ex.max():.1f}')

## 6. Visualise Nu profiles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, Re in zip(axes, [500, 2000, 6000]):
    r_D = np.linspace(0.1, 6, 200)
    for H_D in [3, 6, 10]:
        Nu = martin_nu(r_D, Re, 0.71, H_D, 5.0)
        ax.plot(r_D, Nu, label=f'H/D={H_D}')
    ax.set_title(f'Re={Re}')
    ax.set_xlabel('r/D'); ax.set_ylabel('Nu')
    ax.legend(fontsize=8)

plt.suptitle('Martin (1977) jet impingement Nu profiles — training data', y=1.02)
plt.tight_layout()
plt.savefig(str(HEAT_CKPT_DIR / 'nu_profiles.png'), dpi=120, bbox_inches='tight')
plt.show()

## 7. Normalisation stats

In [ ]:
norm_path = HEAT_CKPT_DIR / 'heat_norm_stats.npz'

if norm_path.exists():
    stats     = np.load(norm_path)
    node_mean = stats['node_mean']; node_std = stats['node_std']
    out_mean  = stats['out_mean'];  out_std  = stats['out_std']
    print(f'Loaded norm stats from {norm_path}')
else:
    sample = train_cases[:200]
    all_nodes = np.concatenate([c['node_input'] for c in sample], axis=0)
    all_outs  = np.concatenate([c['targets']    for c in sample], axis=0)

    node_mean = all_nodes.mean(0).astype(np.float32)
    node_std  = all_nodes.std(0).clip(1e-6).astype(np.float32)
    out_mean  = all_outs.mean(0).astype(np.float32)
    out_std   = all_outs.std(0).clip(1e-6).astype(np.float32)

    np.savez(norm_path, node_mean=node_mean, node_std=node_std,
             out_mean=out_mean, out_std=out_std)
    print(f'Saved norm stats to {norm_path}')

print(f'Output std (Nu, h, T_norm): {out_std.round(2)}')

## 8. Graph builder (same as flow head)

In [ ]:
K_NEIGHBORS = 6

def make_batch(case: dict) -> dict:
    coords     = case['coords']
    node_input = case['node_input']
    targets    = case['targets']

    tree = cKDTree(coords[:, :2])
    _, idx = tree.query(coords[:, :2], k=K_NEIGHBORS+1)
    idx = idx[:, 1:]
    N   = len(coords)
    src = np.repeat(np.arange(N), K_NEIGHBORS)
    dst = idx.flatten()

    diff     = coords[dst] - coords[src]
    dist     = np.linalg.norm(diff, axis=1, keepdims=True)
    med_dist = float(np.median(dist)) + 1e-8
    edge_feats = np.concatenate([diff / med_dist, dist / med_dist], axis=1).astype(np.float32)

    node_norm = ((node_input - node_mean) / node_std).astype(np.float32)
    tgt_norm  = ((targets    - out_mean)  / out_std).astype(np.float32)

    return {
        'x':          torch.from_numpy(node_norm),
        'edge_index': torch.tensor(np.stack([src, dst]), dtype=torch.long),
        'edge_attr':  torch.from_numpy(edge_feats),
        'y':          torch.from_numpy(tgt_norm),
    }

print('Graph builder ready.')

## 9. Phase 1 — Train heat decoder (encoder frozen)

Encoder weights from flow pretraining are frozen. Only the heat decoder trains.

In [ ]:
# Freeze encoder
for name, param in heat_model.named_parameters():
    if 'decoder' not in name:
        param.requires_grad = False

frozen_params   = sum(p.numel() for p in heat_model.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in heat_model.parameters() if p.requires_grad)
print(f'Frozen:    {frozen_params/1e6:.1f}M params')
print(f'Trainable: {trainable_params/1e6:.1f}M params (decoder only)')

optimizer1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, heat_model.parameters()),
    lr=CFG['lr_frozen']
)
scheduler1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer1, T_max=CFG['epochs_frozen'], eta_min=1e-5
)
dtype = torch.bfloat16 if CFG['bf16'] else torch.float32

train_losses1, val_losses1 = [], []

print(f'\nPhase 1: {CFG["epochs_frozen"]} epochs, lr={CFG["lr_frozen"]}')
t0 = time.time()

for epoch in range(CFG['epochs_frozen']):
    heat_model.train()
    np.random.shuffle(train_cases)
    epoch_loss = 0.0
    optimizer1.zero_grad()

    for step, case in enumerate(train_cases):
        b = make_batch(case)
        x          = b['x'].to(DEVICE)
        edge_index = b['edge_index'].to(DEVICE)
        edge_attr  = b['edge_attr'].to(DEVICE)
        targets    = b['y'].to(DEVICE)

        with torch.amp.autocast('cuda', dtype=dtype):
            pred = heat_model(x, edge_index, edge_attr)
            loss = nn.functional.mse_loss(pred, targets)

        loss.backward()
        if (step + 1) % 8 == 0 or step == len(train_cases) - 1:
            torch.nn.utils.clip_grad_norm_(heat_model.parameters(), CFG['grad_clip'])
            optimizer1.step()
            optimizer1.zero_grad()

        epoch_loss += loss.item()
        del x, edge_index, edge_attr, targets, pred, loss

    scheduler1.step()
    train_losses1.append(epoch_loss / len(train_cases))

    if (epoch + 1) % CFG['log_every'] == 0 or epoch == 0:
        heat_model.eval()
        with torch.no_grad():
            vl = 0.0
            for case in val_cases:
                b = make_batch(case)
                pred = heat_model(b['x'].to(DEVICE), b['edge_index'].to(DEVICE), b['edge_attr'].to(DEVICE))
                vl += nn.functional.mse_loss(pred, b['y'].to(DEVICE)).item()
            vl /= len(val_cases)
        val_losses1.append((epoch, vl))
        print(f'  Ep {epoch+1:3d}/{CFG["epochs_frozen"]}  '
              f'train={train_losses1[-1]:.4f}  val={vl:.4f}  '
              f'lr={scheduler1.get_last_lr()[0]:.2e}')

    if (epoch + 1) % CFG['save_every'] == 0:
        torch.save({'epoch': epoch, 'model': heat_model.state_dict(),
                    'train_losses': train_losses1, 'val_losses': val_losses1},
                   HEAT_CKPT_DIR / f'heat_phase1_epoch{epoch+1:04d}.pt')
        print(f'  ✓  Checkpoint saved')

print(f'Phase 1 complete in {(time.time()-t0)/60:.1f} min')

## 10. Phase 2 — Fine-tune encoder + heat decoder

In [ ]:
# Unfreeze encoder with very low LR
for param in heat_model.parameters():
    param.requires_grad = True

optimizer2 = torch.optim.AdamW([
    {'params': heat_model.decoder.parameters(),   'lr': CFG['lr_finetune']},
    {'params': list(heat_model.node_enc.parameters()) +
               list(heat_model.edge_enc.parameters()) +
               list(heat_model.processors.parameters()), 'lr': CFG['lr_encoder']},
], weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer2, T_max=CFG['epochs_finetune'], eta_min=1e-6
)

train_losses2, val_losses2 = [], []

print(f'Phase 2: {CFG["epochs_finetune"]} epochs  '
      f'decoder lr={CFG["lr_finetune"]}  encoder lr={CFG["lr_encoder"]}')
t0 = time.time()

for epoch in range(CFG['epochs_finetune']):
    heat_model.train()
    np.random.shuffle(train_cases)
    epoch_loss = 0.0
    optimizer2.zero_grad()

    for step, case in enumerate(train_cases):
        b = make_batch(case)
        x          = b['x'].to(DEVICE)
        edge_index = b['edge_index'].to(DEVICE)
        edge_attr  = b['edge_attr'].to(DEVICE)
        targets    = b['y'].to(DEVICE)

        with torch.amp.autocast('cuda', dtype=dtype):
            pred = heat_model(x, edge_index, edge_attr)
            loss = nn.functional.mse_loss(pred, targets)

        loss.backward()
        if (step + 1) % 8 == 0 or step == len(train_cases) - 1:
            torch.nn.utils.clip_grad_norm_(heat_model.parameters(), CFG['grad_clip'])
            optimizer2.step()
            optimizer2.zero_grad()

        epoch_loss += loss.item()
        del x, edge_index, edge_attr, targets, pred, loss

    scheduler2.step()
    train_losses2.append(epoch_loss / len(train_cases))

    if (epoch + 1) % CFG['log_every'] == 0 or epoch == 0:
        heat_model.eval()
        with torch.no_grad():
            vl = 0.0
            for case in val_cases:
                b = make_batch(case)
                pred = heat_model(b['x'].to(DEVICE), b['edge_index'].to(DEVICE), b['edge_attr'].to(DEVICE))
                vl += nn.functional.mse_loss(pred, b['y'].to(DEVICE)).item()
            vl /= len(val_cases)
        val_losses2.append((epoch, vl))
        print(f'  Ep {epoch+1:3d}/{CFG["epochs_finetune"]}  '
              f'train={train_losses2[-1]:.4f}  val={vl:.4f}')

    if (epoch + 1) % CFG['save_every'] == 0:
        torch.save({'epoch': epoch, 'model': heat_model.state_dict()},
                   HEAT_CKPT_DIR / f'heat_phase2_epoch{epoch+1:04d}.pt')
        print(f'  ✓  Checkpoint saved')

print(f'Phase 2 complete in {(time.time()-t0)/60:.1f} min')

## 11. Loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses1, label='Phase 1 train')
axes[0].plot(train_losses2, label='Phase 2 train')
if val_losses1:
    ep, vl = zip(*val_losses1); axes[0].plot(ep, vl, 'o--', label='Phase 1 val')
if val_losses2:
    ep, vl = zip(*val_losses2); axes[0].plot(ep, vl, 's--', label='Phase 2 val')
axes[0].set_yscale('log'); axes[0].legend(); axes[0].set_title('Heat Head — Loss Curves')

# Nu prediction vs ground truth on one val case
heat_model.eval()
case = val_cases[0]
b    = make_batch(case)
with torch.no_grad():
    pred = heat_model(b['x'].to(DEVICE), b['edge_index'].to(DEVICE), b['edge_attr'].to(DEVICE)).cpu().numpy()

Nu_pred = pred[:, 0] * out_std[0] + out_mean[0]
Nu_true = case['targets'][:, 0]
r_D     = np.linalg.norm(case['coords'][:, :2], axis=1)
sort_idx = np.argsort(r_D)

axes[1].plot(r_D[sort_idx], Nu_true[sort_idx], label='True Nu', alpha=0.7)
axes[1].plot(r_D[sort_idx], Nu_pred[sort_idx], label='Pred Nu', alpha=0.7, linestyle='--')
axes[1].set_xlabel('r/D'); axes[1].set_ylabel('Nu')
axes[1].set_title(f'Nu prediction  Re={case["Re"]:.0f}, H/D={case["H_D"]:.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(str(HEAT_CKPT_DIR / 'heat_loss_curves.png'), dpi=120)
plt.show()

## 12. Save final heat checkpoint

In [ ]:
final_path = HEAT_CKPT_DIR / 'heat_head_final.pt'
torch.save({
    'model': heat_model.state_dict(),
    'cfg':   CFG,
    'norm':  {
        'node_mean': node_mean.tolist(), 'node_std': node_std.tolist(),
        'out_mean':  out_mean.tolist(),  'out_std':  out_std.tolist(),
    },
    'final_train_loss': train_losses2[-1] if train_losses2 else train_losses1[-1],
    'final_val_loss':   val_losses2[-1][1] if val_losses2 else val_losses1[-1][1],
}, final_path)

print(f'Final checkpoint → {final_path}')
print(f'Final train MSE  : {train_losses2[-1]:.4f}')
print(f'Final val MSE    : {val_losses2[-1][1]:.4f}')
print('\n✓  Heat head complete — ready for 04_species_head.ipynb')